# FORGE — Walkthrough su dati reali AMZN 1D

Questo notebook percorre **l'intero pipeline FORGE** su 5 anni di candele giornaliere di Amazon (Jan 2021 → Jun 2026).

```
Candele OHLCV
  └─ build_features()      Feature engineering (EMA, RSI, BB, ATR, MACD, …)
       └─ forge()           Pipeline completa M0 → M1 → M2 → M3
            ├─ M0  MarketContext      — regime di mercato (bull/bear/…)
            ├─ M1  EventDiscovery     — segnali atomici consistenti
            ├─ M2  AlphaDiscovery     — contratti con direzione + holding period
            └─ M3  RuleDiscovery      — validazione IS + walk-forward OOS
```

**Output finale:** regole di trading statisticamente robuste (PARTIAL-EDGE / EDGE) già backtestabili.

## 1 — Setup

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter

from forgedge import build_features, forge, forge_preset

DATA = Path("data/AMZN_1D.csv")

## 2 — Caricamento e pulizia dati

Il file CSV viene da Investing.com (formato standard: Date, Price, Open, High, Low, Vol., Change %).
Le colonne vengono rinominate e il volume viene convertito da formato `"31.50M"` a float.

In [2]:
raw = pd.read_csv(DATA)

# normalise column names: lower, no dots/spaces
raw.columns = [c.strip().lower().replace(".", "").replace(" ", "_") for c in raw.columns]

# rename to OHLCV convention
raw = raw.rename(columns={"price": "close", "vol": "volume", "change_%": "chg_pct"})

# parse date and sort ascending
raw["open_dt"] = pd.to_datetime(raw["date"], format="%m/%d/%Y")
raw = raw.sort_values("open_dt").reset_index(drop=True)

# numeric price columns
for col in ["open", "high", "low", "close"]:
    raw[col] = pd.to_numeric(
        raw[col].astype(str).str.replace(",", ""), errors="coerce"
    )

# volume: "31.50M" → 31_500_000 ; "1.2B" → 1_200_000_000
raw["volume"] = (
    raw["volume"]
    .astype(str)
    .str.replace("M", "e6")
    .str.replace("B", "e9")
)
raw["volume"] = pd.to_numeric(raw["volume"], errors="coerce").fillna(0)

# integer unix-ms timestamp required by build_features
raw["open_time"] = raw["open_dt"].astype("datetime64[ms]").astype("int64")

candles = raw[["open_time", "open_dt", "open", "high", "low", "close", "volume"]]

print(f"Barre: {len(candles)}")
print(f"Periodo: {candles['open_dt'].iloc[0].date()} → {candles['open_dt'].iloc[-1].date()}")
candles.tail(3)

Barre: 1378
Periodo: 2021-01-04 → 2026-06-30


,open_time,open_dt,open,high,low,close,volume
1375,1782432000000,2026-06-26,227.21,233.90,226.13,232.69,248370000.0
1376,1782691200000,2026-06-29,234.22,249.71,233.80,240.14,77620000.0
1377,1782777600000,2026-06-30,237.50,241.53,237.57,238.71,31500000.0


## 3 — Feature engineering

`build_features()` calcola indicatori tecnici sulle candele e restituisce la **KPI Table**: la struttura che serve a `forge()`. Ogni indicatore è descritto da un blocco di configurazione.

In [3]:
indicator_config = {
    "ema": {
        "enabled": True,
        "params": {"periods": [9, 50], "columns": ["close"]},
    },
    "rsi": {
        "enabled": True,
        "params": {"periods": [14, 25], "columns": ["close"]},
    },
    "bollinger_bands": {
        "enabled": True,
        "params": {"periods": [20], "columns": ["close"]},
    },
    "atr": {
        "enabled": True,
        "params": {"periods": [14], "columns": ["close"]},
    },
    "macd": {
        "enabled": True,
        "params": {"fast": 12, "slow": 26, "signal": 9, "columns": ["close"]},
    },
}

kpi = build_features(candles, indicator_config, timestamp_col="open_time")

feature_cols = [c for c in kpi.columns if c not in candles.columns]
print(f"KPI Table: {kpi.shape[0]} righe × {kpi.shape[1]} colonne")
print(f"Feature generate: {len(feature_cols)}")
print("Esempi:", ", ".join(feature_cols[:10]), "…")

indicatore 'atr' non implementato, salto


indicatore 'macd' non implementato, salto


KPI Table: 1378 righe × 16 colonne
Feature generate: 9
Esempi: color, close_ema_09, close_ema_50, close_rsi_14, close_rsi_25, close_bb_mid_20, close_bb_upper_20, close_bb_lower_20, close_bb_width_20 …


## 4 — Configurazione preset

`forge_preset()` restituisce una tripletta di config pre-calibrate per il timeframe e l'asset.

| Preset | Filosofia | Copertura |
|--------|-----------|----------|
| **sniper** | solo segnali ad alta coerenza temporale | bassa |
| **balanced** | equilibrio qualità/copertura | media |
| **sweep** | esplora tutto, filtra poco | alta |
| **burst** | segnali ad alta frequenza di attivazione | media |

Per AMZN 1D usiamo **sweep** per massimizzare la copertura esplorativa.

In [4]:
event_cfg, alpha_cfg, rd_cfg = forge_preset(
    "sweep",
    timeframe="1D",
    asset="AMZN",
)

gp = event_cfg.gate_params
sc = rd_cfg.criteria
print(f"Gate: min_tpm={gp.min_tpm:.2f} ep/mese  max_dispersion={gp.max_dispersion}  mode='{gp.event_counting}'")
print(f"RD:   min_profit_factor={sc.min_profit_factor}  min_tpm={sc.min_tpm}")

Gate: min_tpm=0.30 ep/mese  max_dispersion=2.5  mode='episode'
RD:   min_profit_factor=2.0  min_tpm=0.3


## 5 — Esecuzione FORGE

Un'unica chiamata `forge()` esegue l'intero pipeline M0 → M3.

In [5]:
result = forge(
    kpi,
    ticker="AMZN",
    timeframe="1D",
    event_discovery_config=event_cfg,
    alpha_config=alpha_cfg,
    rule_discovery_config=rd_cfg,
    run_registry=False,
    progress=True,
)

print(f"\nM1 candidati  : {len(result.candidates)}")
print(f"M2 contratti  : {len(result.contracts)}")
print(f"M2 promossi   : {len(result.promoted)}")
print(f"M3 risposte   : {len(result.rule_responses)}")

[forge:AMZN +   0.0s] start — 1378 bars


[forge:AMZN +   0.0s] M0 Market Context — classifying regimes…


[forge:AMZN +   0.1s] M1 Event Discovery — mining event candidates…


[forge:AMZN +   6.5s] M1 Event Discovery — 2717 candidate(s)


[forge:AMZN +   6.5s] M2 Alpha Discovery — evaluating 2717 candidate(s)…


...

[forge:AMZN] M3 Rule Discovery |███████████████████████████·| 404/405 (100%) 135.7s

[forge:AMZN] M3 Rule Discovery |████████████████████████████| 405/405 (100%) 135.8s

[forge:AMZN + 151.8s] M3 Rule Discovery — 38 tradeable rule(s)


[forge:AMZN + 151.8s] done



M1 candidati  : 2717
M2 contratti  : 2717
M2 promossi   : 405
M3 risposte   : 405


## 6 — M1: Event Discovery

M1 genera **candidati atomici** (espressioni soglia `feature < valore`) filtrati dalla **ConsistencyGate** in base a frequenza mensile (`min_tpm`) e regolarità temporale (`max_dispersion` — indice di dispersione di Poisson). I candidati che superano il gate vengono composti in **AND** per formare segnali multivariati.

In [6]:
candidates = result.candidates
passed = [c for c in candidates if c.consistency_gate.passed]

print(f"Candidati totali : {len(candidates)}")
print(f"Gate superato    : {len(passed)}")
print(f"Atomici (1 feat) : {sum(1 for c in candidates if len(c.components) == 1)}")
print(f"AND composti     : {sum(1 for c in candidates if len(c.components) > 1)}")

Candidati totali : 2717
Gate superato    : 2717
Atomici (1 feat) : 717
AND composti     : 2000


In [7]:
# Top 10 per episodi/mese
rows = []
for c in sorted(candidates, key=lambda x: x.activation_stats.mean_tpm, reverse=True)[:10]:
    s = c.activation_stats
    rows.append({
        "event_id":   c.event_id,
        "expression": c.expression,
        "tpm":        round(s.mean_tpm, 3),
        "dispersion": round(s.index_of_dispersion, 3),
        "n_ep":       s.n_activations,
        "gate":       "✓" if c.consistency_gate.passed else "✗",
    })

pd.DataFrame(rows)

,event_id,expression,tpm,dispersion,n_ep,gate
0,EVT-ratio_close_ema09_em-ZS-0238,zs_ratio_close_ema09_ema50_48 < -1,5.439,8.520,359,✓
1,EVT-color-ZS-0025,zs_color_168 > 1,5.273,4.432,348,✓
2,EVT-diffnorm_close_ema09-ZS-0266,zs_diffnorm_close_ema09_ema50_48 > 1,5.212,8.699,344,✓
3,EVT-diffnorm_close_ema09-ZS-0265,zs_diffnorm_close_ema09_ema50_48 < -1,5.136,8.410,339,✓
4,EVT-ratio_close_ema09_em-ZS-0239,zs_ratio_close_ema09_ema50_48 > 1,4.985,8.342,329,✓
5,EVT-close_rsi_25-ZS-0140,zs_close_rsi_25_48 > 1,4.955,7.467,327,✓
6,EVT-close_bb_width_20-ZS-0206,zs_close_bb_width_20_48 > 1,4.939,6.465,326,✓
7,EVT-diffnorm_close_ema09-ZS-0268,zs_diffnorm_close_ema09_ema50_96 > 1,4.864,9.875,321,✓
8,EVT-spread_close_ema50-ZS-0573,zs_spread_close_ema50_48 < -1,4.742,6.750,313,✓
9,EVT-ratio_close_rsi14_rs-ZS-0322,zs_ratio_close_rsi14_rsi25_48 < -1,4.667,4.473,308,✓


## 7 — M2: Alpha Discovery

Per ogni candidato M1 promosso, M2 ricerca il **contratto ottimale**:
- **direzione** (long / short)
- **holding period h\*** (da 1 a `max_holding_period` barre)
- **sell_pct**: percentuale di gain target

Vengono promossi solo i contratti con `lift` statisticamente significativo.

In [8]:
promoted = result.promoted

print(f"Contratti totali M2 : {len(result.contracts)}")
print(f"Promossi            : {len(promoted)}")
print(f"  long              : {sum(1 for c in promoted if c.direction == 'long')}")
print(f"  short             : {sum(1 for c in promoted if c.direction == 'short')}")

Contratti totali M2 : 2717
Promossi            : 405
  long              : 204
  short             : 201


In [9]:
# Top 10 contratti promossi per lift
rows = []
for c in sorted(promoted, key=lambda x: x.event_stats.lift, reverse=True)[:10]:
    dt = c.derived_target
    rows.append({
        "expression":  c.event_expression,
        "direction":   c.direction,
        "h*":          dt.holding_period_h,
        "sell_pct %":  round(dt.sell_pct * 100, 2),
        "lift":        round(c.event_stats.lift, 4),
        "activations": c.event_stats.n_activations,
    })

pd.DataFrame(rows)

,expression,direction,h*,sell_pct %,lift,activations
0,delta_close_rsi_14_1 > 4.46123 AND close_rsi_2...,long,7,7.70,0.6952,15
1,zs_color_48 < -1 AND pr_close_rsi_14_168 < 0.1...,long,3,3.58,0.6562,11
2,zs_color_48 < -1 AND zs_close_rsi_14_168 < -1,long,3,3.72,0.5909,12
3,close_rsi_14 < 39.2903 AND delta_ratio_close_r...,long,10,7.89,0.5735,17
4,close_rsi_14 < 36.3905 AND delta_bb_pct_b_clos...,long,2,2.18,0.5385,18
5,close_rsi_14 < 36.3905 AND delta_spread_close_...,long,1,1.25,0.5253,22
6,close_rsi_25 < 40.3848,long,10,7.94,0.5219,96
7,delta_close_rsi_14_12 < -24.9697,long,2,1.99,0.5188,24
8,close_rsi_25 < 39.2649,long,10,9.34,0.5150,76
9,pr_color_168 < 0.247024 AND bb_pct_b_close_20 ...,long,2,2.11,0.5125,16


## 8 — M3: Rule Discovery

M3 valida ogni contratto promosso con:
1. **Backtest IS** — profit factor, win rate, expectancy
2. **Walk-forward OOS** — 4 split espandenti; consistency e degradazione
3. **Validazione statistica** — Deflated Sharpe, t-test win rate

Verdetti:
- `EDGE` — robusto IS + OOS
- `PARTIAL-EDGE` — IS forte, OOS sufficiente ma non pieno
- `NON-EDGE` — non supera i criteri

In [10]:
verdicts = Counter(r.verdict for _, r in result.rule_responses)
print("Verdetti M3:")
for v, n in sorted(verdicts.items(), key=lambda x: -x[1]):
    bar = "█" * n
    print(f"  {v:<16} {n:>4}  {bar}")

Verdetti M3:
  NON-EDGE          367  ███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  PARTIAL-EDGE       38  ██████████████████████████████████████


## 9 — Tabella PARTIAL-EDGE

Tutti i segnali con verdetto `PARTIAL-EDGE`, ordinati per profit factor IS.

In [11]:
partial_edge = [
    (cc, r)
    for cc, r in result.rule_responses
    if r.verdict == "PARTIAL-EDGE"
]
partial_edge.sort(key=lambda x: x[1].in_sample_summary.profit_factor, reverse=True)

rows = []
for cc, r in partial_edge:
    iss = r.in_sample_summary
    wf  = r.walk_forward
    oos = wf.oos_summary if wf else None
    dt  = cc.derived_target
    rows.append({
        "expression":  cc.event_expression,
        "dir":         cc.direction,
        "h*":          dt.holding_period_h if dt else None,
        "IS PF":       round(iss.profit_factor, 2),
        "IS WR %":     round(iss.win_rate_pct * 100, 1),
        "IS trades":   iss.total_trades,
        "OOS PF":      round(oos.profit_factor, 2) if oos and oos.total_trades else None,
        "OOS WR %":    round(oos.win_rate_pct * 100, 1) if oos and oos.total_trades else None,
        "OOS trades":  oos.total_trades if oos else None,
        "consistency": f"{wf.n_profitable_splits}/{len(wf.splits)}" if wf else None,
    })

df_pe = pd.DataFrame(rows)
print(f"{len(df_pe)} segnali PARTIAL-EDGE")
df_pe

38 segnali PARTIAL-EDGE


,expression,dir,h*,IS PF,IS WR %,IS trades,OOS PF,OOS WR %,OOS trades,consistency
0,zs_color_168 > 1 AND delta_ratio_close_ema09_e...,short,5,15.26,95.2,21,1.21,76.2,21,3/4
1,delta_close_rsi_14_1 > 4.46123 AND close_rsi_2...,short,10,8.69,95.2,21,3.52,75.0,20,4/4
2,pr_color_168 > 0.761905 AND delta_ratio_close_...,short,7,7.19,75.0,20,1.55,68.4,19,2/4
3,zs_color_168 > 1 AND delta_close_rsi_25_6 < -6...,short,5,6.72,90.5,21,4.19,89.5,19,4/4
4,zs_color_168 > 1 AND delta_ratio_close_ema09_e...,short,7,6.51,80.0,25,1.69,71.4,28,3/4
5,delta_close_rsi_14_3 > 8.10394 AND zs_close_bb...,short,3,6.27,82.6,23,3.94,80.0,20,3/4
6,delta_close_rsi_14_1 < -4.52196 AND delta_clos...,short,5,5.21,77.8,27,3.52,81.2,16,3/4
7,zs_color_168 > 1 AND pr_ratio_close_rsi14_rsi2...,short,7,4.12,95.0,20,1.37,77.3,22,3/4
8,zs_color_168 > 1 AND delta_ratio_close_rsi14_r...,short,7,3.98,70.0,20,1.22,55.6,18,3/4
9,zs_color_96 > 1 AND delta_ratio_close_rsi14_rs...,short,10,3.97,63.2,19,1.79,62.5,16,3/4


## 10 — Dettaglio sui segnali con 100% consistency

I segnali profittevoli in **tutti gli split OOS** sono i candidati più robusti per il paper trading.

In [12]:
def print_signal_detail(cc, r, rank):
    iss = r.in_sample_summary
    wf  = r.walk_forward
    oos = wf.oos_summary if wf else None
    dt  = cc.derived_target
    sv  = r.statistical_validation
    wfv = wf.oos_validation if wf else None

    print(f"{'═'*72}")
    print(f"  #{rank}  {r.verdict}")
    print(f"  Regola:  {cc.event_expression}")
    if dt:
        print(f"  Setup:   {cc.direction.upper()}  h*={dt.holding_period_h} barre  target={dt.sell_pct*100:.2f}%")
    print()
    print(f"  ┌─ IN-SAMPLE ({iss.total_trades} trades) ")
    print(f"  │  PF={iss.profit_factor:.3f}  WR={iss.win_rate_pct*100:.1f}%  Expect={iss.expectancy*100:.2f}%  TPM={iss.tpm_mu:.2f}")
    if sv:
        print(f"  │  DSharpe={sv.deflated_sharpe:.2f}  WR t={sv.ttest_winrate_t:.2f}  "
              f"trials={sv.n_trials_tested}  stability={sv.temporal_stability}")

    if oos and oos.total_trades:
        consistency = f"{wf.n_profitable_splits}/{len(wf.splits)}"
        print(f"  ├─ OOS AGGREGATO ({oos.total_trades} trades, {consistency} split profittevoli)")
        print(f"  │  PF={oos.profit_factor:.3f}  WR={oos.win_rate_pct*100:.1f}%  Expect={oos.expectancy*100:.2f}%  TPM={oos.tpm_mu:.2f}")
        if wfv:
            print(f"  │  stability={wfv.temporal_stability}  PF½1={wfv.pf_first_half:.2f}  PF½2={wfv.pf_second_half:.2f}")

    if wf and wf.splits:
        print(f"  ├─ WALK-FORWARD SPLITS")
        for sp in wf.splits:
            ts = sp.test_summary
            ok = "✓" if ts.profit_factor > 1 else "✗"
            pf_str = f"{ts.profit_factor:.2f}" if ts.profit_factor < 9999 else "∞"
            print(f"  │  [{ok}] {sp.test_from[:10]}→{sp.test_to[:10]}  "
                  f"PF={pf_str:<6}  WR={ts.win_rate_pct*100:.0f}%  t={ts.total_trades}")

    print(f"  └{'─'*69}")
    print()

In [13]:
robust = [
    (cc, r)
    for cc, r in partial_edge
    if r.walk_forward
    and r.walk_forward.n_profitable_splits == len(r.walk_forward.splits)
    and r.walk_forward.n_profitable_splits > 0
]

print(f"Segnali con consistency 100%: {len(robust)}\n")
for i, (cc, r) in enumerate(robust, 1):
    print_signal_detail(cc, r, i)

Segnali con consistency 100%: 2

════════════════════════════════════════════════════════════════════════
  #1  PARTIAL-EDGE
  Regola:  delta_close_rsi_14_1 > 4.46123 AND close_rsi_25 > 61.0731
  Setup:   SHORT  h*=10 barre  target=3.49%

  ┌─ IN-SAMPLE (21 trades) 
  │  PF=8.689  WR=95.2%  Expect=1.76%  TPM=0.32
  │  DSharpe=4.41  WR t=11.36  trials=75  stability=WARN
  ├─ OOS AGGREGATO (20 trades, 4/4 split profittevoli)
  │  PF=3.515  WR=75.0%  Expect=1.05%  TPM=0.33
  │  stability=WARN  PF½1=1.09  PF½2=15.62
  ├─ WALK-FORWARD SPLITS
  │  [✓] 2021-07-01→2022-10-01  PF=18.32   WR=50%  t=2
  │  [✓] 2022-10-01→2024-01-01  PF=1.23    WR=86%  t=7
  │  [✓] 2024-01-01→2025-04-01  PF=4.50    WR=57%  t=7
  │  [✓] 2025-04-01→2026-07-01  PF=∞       WR=100%  t=4
  └─────────────────────────────────────────────────────────────────────

════════════════════════════════════════════════════════════════════════
  #2  PARTIAL-EDGE
  Regola:  zs_color_168 > 1 AND delta_close_rsi_25_6 < -6.39394
  Setu

## 11 — Interpretazione

### Segnali chiave trovati da FORGE su AMZN 1D

**`zs_color_168 > 1 AND delta_close_rsi_25_6 < -6.39` — SHORT h\*=5**
- `zs_color_168 > 1`: momentum di mercato ribassista su 168 barre (le barre negative sono sopra la media z-score)
- `delta_close_rsi_25_6 < -6.39`: RSI25 sceso di almeno 6 punti in 6 sedute → segnale di accelerazione short
- **OOS**: PF ≈ 4.2, WR 89%, 100% consistency → stabile nel tempo

**`delta_close_rsi_14_1 > 4.46 AND close_rsi_25 > 61.07` — SHORT h\*=10**
- RSI14 spike giornaliero ≥ 4.5 punti mentre RSI25 è in zona ipercomprata (>61)
- Divergenza momentum breve / lungo termine → alta probabilità di correzione
- **OOS**: PF ≈ 3.5, WR 75%, 100% consistency

### Note sul verdetto PARTIAL-EDGE
- `PARTIAL-EDGE` ≠ segnale invalido: indica che l'edge c'è ma non ha ancora la significatività statistica piena
- Passo consigliato: paper trading 3-6 mesi, poi rivalutare con `EDGE` se il PF OOS live ≥ threshold
- Monitorare il **decay** del PF: se il PF OOS scende sotto 1.5 in live, il regime è cambiato

---

## Appendice — Struttura di `ForgeResult`

```python
result = forge(kpi, ...)

result.enriched          # KPI Table dopo MarketContext (+ colonna 'regime')
result.candidates        # List[EventCandidate]  — tutti i candidati M1
result.contracts         # List[AlphaContract]   — tutti i contratti M2
result.promoted          # List[AlphaContract]   — solo quelli promossi
result.rule_responses    # List[Tuple[AlphaContract, RuleDiscoveryResponse]]

# Per ogni risposta M3:
cc, r = result.rule_responses[0]
r.verdict                       # "EDGE" / "PARTIAL-EDGE" / "NON-EDGE"
r.in_sample_summary.profit_factor
r.in_sample_summary.win_rate_pct  # fraction [0, 1]
r.in_sample_summary.total_trades
r.in_sample_summary.expectancy
r.walk_forward.oos_summary        # BacktestSummary aggregato OOS
r.walk_forward.consistency        # frazione split profittevoli
r.walk_forward.splits             # List[WalkForwardSplit]
r.statistical_validation          # DSharpe, t-stat, temporal_stability

# Dal contratto M2:
cc.event_expression               # regola leggibile
cc.direction                      # "long" / "short"
cc.derived_target.holding_period_h
cc.derived_target.sell_pct
cc.event_stats.lift
cc.event_stats.n_activations
```